In [ ]:
# Chapter 10: Training Deep Neural Networks

## The Vanishing and Exploding Gradients Problems

Training deep neural networks is difficult because gradients can become unstable as they propagate backward through the network. They often get smaller and smaller (vanishing gradients) or larger and larger (exploding gradients), making lower layers hard to train. This instability is often caused by the combination of the logistic sigmoid activation function and random weight initialization using a normal distribution.

<p align="left"><img src="../fig/figure11.1.png" width="45%"></p>

### Glorot and He Initialization

To ensure signals flow properly without vanishing or exploding, the variance of the outputs of each layer should equal the variance of its inputs. Glorot (Xavier) and He initialization strategies achieve this by randomizing weights based on the number of inputs ($fan_{in}$) and outputs ($fan_{out}$) of the layer.
- Glorot Initialization: Used for None, tanh, logistic, and softmax functions.

- He Initialization: Used for ReLU and its variants.

- LeCun Initialization: Used for SELU.

By default, Keras uses Glorot initialization. You can change it to He initialization for ReLU layers:

In [1]:
import tensorflow as tf
from tensorflow import keras

# Using He Initialization for a Dense layer with ReLU
layer = keras.layers.Dense(10, activation="relu", kernel_initializer="he_normal")

### Nonsaturating Activation Functions

The logistic and tanh functions saturate at their extremes. ReLU is better but suffers from "dying ReLUs".

<p align="left"><img src="../fig/figure11.2.png" width="45%"></p>

- Leaky ReLU: Has a small slope for negative values (the leak) to keep neurons alive.

In [2]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.Sequential([
    # ... other layers ...
    keras.layers.Dense(10, kernel_initializer="he_normal"),
    keras.layers.LeakyReLU(alpha=0.2), # Add LeakyReLU as a separate layer
    # ... other layers ...
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


- SELU (Scaled ELU): Enables self-normalization in deep networks of dense layers, solving the vanishing gradient problem naturally if input features are standardized.

In [3]:
import tensorflow as tf
from tensorflow import keras

# SELU activation with LeCun Normal initialization
layer = keras.layers.Dense(10, activation="selu",
                           kernel_initializer="lecun_normal")

### Batch Normalization
Batch Normalization (BN) addresses unstable gradients by zero-centering and normalizing inputs, then scaling and shifting them. It allows for higher learning rates and acts as a regularizer.

In [4]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Gradient Clipping
Useful primarily for Recurrent Neural Networks (RNNs) to prevent exploding gradients.

In [5]:
import tensorflow as tf
from tensorflow import keras

# Clips values between -1.0 and 1.0
optimizer = keras.optimizers.SGD(clipvalue=1.0)
model.compile(loss="mse", optimizer=optimizer)

## Reusing Pretrained Layers (Transfer Learning)
Reuse the lower layers of existing networks trained on similar tasks to speed up training and reduce data requirements.

<p align="left"><img src="../fig/figure11.4.png" width="45%"></p>

### Implementation in Keras
1. Load model A and create model B using A's layers (excluding the output).

In [6]:
import tensorflow as tf
from tensorflow import keras

# Assuming 'my_model_A.h5' exists
# model_A = keras.models.load_model("my_model_A.h5")

# For demonstration, let's create a dummy model A
model_A = keras.models.Sequential([keras.layers.Dense(10, input_shape=[5])])

# Create Model B reusing layers from A (excluding the last one)
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2. Clone model A if you want to preserve it, as training B updates A's weights.

In [7]:
import tensorflow as tf
from tensorflow import keras

# Dummy model A for context
model_A = keras.models.Sequential([keras.layers.Dense(10, input_shape=[5])])

# Clone architecture and copy weights
model_A_clone = keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

3. Freeze reused layers initially.

In [8]:
import tensorflow as tf
from tensorflow import keras

# Dummy setup
model_A = keras.models.Sequential([keras.layers.Dense(10, input_shape=[5])])
model_B_on_A = keras.models.Sequential(model_A.layers)

# Freeze layers
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

model_B_on_A.compile(loss="binary_crossentropy", optimizer="sgd",
                     metrics=["accuracy"])

## Faster Optimizers
Standard Gradient Descent is slow. These optimizers speed up training.
- Momentum Optimization:

In [9]:
import tensorflow as tf
from tensorflow import keras

optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

- Nesterov Accelerated Gradient (NAG):

In [10]:
import tensorflow as tf
from tensorflow import keras

optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True)

- RMSProp:

In [11]:
optimizer = keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

- Adam: Combines Momentum and RMSProp.

In [12]:
import tensorflow as tf
from tensorflow import keras

optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

## Learning Rate Scheduling
Start with a high rate and reduce it over time.
- Power Scheduling:

In [13]:
import tensorflow as tf
from tensorflow import keras

# decay is the inverse of 's' (steps)
optimizer = keras.optimizers.SGD(learning_rate=0.01, decay=1e-4)

/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


- Exponential Scheduling:

In [14]:
import tensorflow as tf
from tensorflow import keras

def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1**(epoch / s)
    return exponential_decay_fn

exponential_decay_fn = exponential_decay(lr0=0.01, s=20)
lr_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay_fn)

# Usage in fit:
# history = model.fit(..., callbacks=[lr_scheduler])

- Performance Scheduling:

In [15]:
import tensorflow as tf
from tensorflow import keras

# Reduce LR when validation loss plateaus
lr_scheduler = keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)

- tf.keras Schedules:

In [16]:
import tensorflow as tf
from tensorflow import keras

# Updates LR at every step, not just every epoch
learning_rate = keras.optimizers.schedules.ExponentialDecay(0.01, 20*100, 0.1) # assuming 100 steps/epoch
optimizer = keras.optimizers.SGD(learning_rate)

## Avoiding Overfitting Through Regularization
- $\ell_1$ and $\ell_2$ Regularization:

In [17]:
import tensorflow as tf
from tensorflow import keras
from functools import partial

# Using partial to avoid repetition
RegularizedDense = partial(keras.layers.Dense,
                           activation="elu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=keras.regularizers.l2(0.01))

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    RegularizedDense(300),
    RegularizedDense(100),
    RegularizedDense(10, activation="softmax", kernel_initializer="glorot_uniform")
])

- Dropout:

In [18]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax")
])

- Monte Carlo (MC) Dropout:

In [19]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Assume 'model' and 'X_test_scaled' are defined
# Forces training=True during prediction to keep dropout active
# y_probas = np.stack([model(X_test_scaled, training=True)
#                      for sample in range(100)])
# y_proba = y_probas.mean(axis=0)

- Max-Norm Regularization:

In [20]:
import tensorflow as tf
from tensorflow import keras

keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal",
                   kernel_constraint=keras.constraints.max_norm(1.))

<Dense name=dense_15, built=False>

## Summary and Practical Guidelines
Default DNN Configuration:
- Initializer: He initialization
- Activation: ELU
- Normalization: Batch Normalization (if deep)
- Regularization: Early Stopping (+ $\ell_2$ if needed)
- Optimizer: Momentum (or RMSProp/Nadam)
- Schedule: 1cycle

Self-Normalizing Net Configuration:

- Initializer: LeCun initialization
- Activation: SELU
- Normalization: None (use self-normalization)
- Regularization: Alpha dropout